# Lab 4 — Why It Matters for Agents: Do You Even Need a Model?

**AIEWF 2026 Workshop — Vector → Hybrid → Do You Even Need a Model?**

This notebook wires the Lab 3 hybrid retriever to an LLM synthesis call.

The thesis: **retrieval quality — not model quality — determines answer quality.**

---

## Requirements
```
# requirements.txt
elasticsearch>=8.17.0
anthropic>=0.28.0
```

Environment variables needed:
- `ES_ENDPOINT` — your Elastic Serverless project URL
- `ES_API_KEY` — your Elastic API key
- `ANTHROPIC_API_KEY` — your Anthropic API key (for synthesis)

In [ ]:
# Cell 1 — Install and Import

# Uncomment to install if needed:
# !pip install elasticsearch anthropic

import os
import json
from elasticsearch import Elasticsearch
import anthropic

print('Imports OK')

In [ ]:
# Cell 2 — Connect to Elasticsearch
# Uses the same Serverless project as Labs 1-3.

ES_ENDPOINT = os.environ.get('ES_ENDPOINT')
ES_API_KEY = os.environ.get('ES_API_KEY')

if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        'Set ES_ENDPOINT and ES_API_KEY environment variables.\n'
        'In Instruqt: these are pre-configured in the sandbox environment.\n'
        'Running locally: export ES_ENDPOINT=https://... ES_API_KEY=...'
    )

es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=60)

info = es.info()
print(f'Connected to Elasticsearch {info["version"]["number"]}')

# Verify corpus is indexed
count = es.count(index='aiewf-workshop-docs')['count']
print(f'Index has {count} documents')

In [ ]:
# Cell 3 — The Lab 3 Hybrid Retriever as a Python Function
#
# Same RRF retriever DSL from Lab 3 Dev Console, now as Python.
# ES client's search() takes the same body dict you'd paste into Dev Console.

INDEX_NAME = 'aiewf-workshop-docs'

def hybrid_search(query: str, size: int = 5) -> list[dict]:
    """
    Run a hybrid RRF search combining BM25 + semantic.
    
    Uses the same RRF retriever built in Lab 3.
    Returns a list of dicts: {id, title, url, body (truncated), score}
    """
    response = es.search(
        index=INDEX_NAME,
        body={
            "retriever": {
                "rrf": {
                    "retrievers": [
                        {
                            "standard": {
                                "query": {
                                    "multi_match": {
                                        "query": query,
                                        "fields": ["title^3", "body"],
                                        "type": "best_fields"
                                    }
                                }
                            }
                        },
                        {
                            "standard": {
                                "query": {
                                    "semantic": {
                                        "field": "body_semantic",
                                        "query": query
                                    }
                                }
                            }
                        }
                    ],
                    "rank_constant": 60,
                    "rank_window_size": 100
                }
            },
            "size": size,
            "_source": ["id", "title", "url", "body"]
        }
    )
    
    results = []
    for hit in response['hits']['hits']:
        src = hit['_source']
        results.append({
            'id': src['id'],
            'title': src['title'],
            'url': src.get('url', ''),
            'body': src['body'][:800],  # truncate for context window management
            'score': hit['_score']
        })
    return results


# Quick test
results = hybrid_search("user can't log in", size=3)
print(f'Hybrid search returned {len(results)} results:')
for r in results:
    print(f'  [{r["id"]}] {r["title"]} (score: {r["score"]:.4f})')

In [ ]:
# Cell 4 — LLM Synthesis Function
#
# Uses claude-haiku-4-5-20251001 — cheapest Claude model. Deliberate choice:
# even a cheap model synthesizes a clear answer when context is correct (Cell 5).
# It fails when context is wrong (Cell 6) — same model, same prompt.
#
# synthesize() uses `anthropic_client` — initialized in Cell 7 (instructor demo).
# Cells 5 and 6 show pre-baked output — no API key needed to see the demo.

def synthesize(context_docs: list[dict], question: str, model: str = 'claude-haiku-4-5-20251001') -> str:
    context_parts = []
    for i, doc in enumerate(context_docs, 1):
        context_parts.append(f"[Document {i}: {doc['title']}]\n{doc['body']}\n")
    context_text = '\n---\n'.join(context_parts)
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=512,
        system=(
            'You are a helpful Elasticsearch documentation assistant. '
            'Answer the question using ONLY the provided documentation context. '
            'If the context does not contain enough information, say so clearly.'
        ),
        messages=[{'role': 'user', 'content': f'Context:\n{context_text}\n\nQuestion: {question}'}]
    )
    return response.content[0].text

print('synthesize() defined — anthropic_client initialized in Cell 7 (instructor demo)')


## Cell 5 — Good Context Demo (PRE-BAKED)

This cell uses a **hardcoded query** and **pre-retrieved good context** (the top hybrid results for "user can't log in", including the SAML authentication troubleshooting page — doc-001).

**Run it as-is. Do not modify the context.**

The context is pre-baked so this demo is deterministic — we proved it works before the workshop.

### Why this works:
The hybrid retriever delivers the right docs. The SAML troubleshooting page (doc-001) explains authentication failures, realm configuration, and credential validation in detail. The model synthesizes it into a clear answer. Even cheap Haiku does this well because the context is good.

In [ ]:
# Cell 5 — GOOD CONTEXT DEMO (pre-baked)
# DO NOT MODIFY — context is hardcoded for reproducibility

DEMO_QUESTION = "A user can't log in. What should I check?"

# Pre-retrieved good context from the hybrid retriever
# (These are real excerpts from the workshop corpus — verified to produce a good answer)
GOOD_CONTEXT = [
    {
        "id": "doc-001",
        "title": "SAML authentication troubleshooting",
        "url": "https://www.elastic.co/docs/troubleshoot/elasticsearch/security/saml",
        "body": (
            "When users experience authentication failure, start by checking the realm configuration. "
            "The most common causes of credential error are: (1) the xpack.security.authc.realms "
            "configuration is missing or incorrect in elasticsearch.yml; (2) the IdP metadata URL "
            "is unreachable or returning stale metadata; (3) the principal attribute mapping does not "
            "match a user in the Elasticsearch user store. \n\n"
            "To diagnose authentication failure: enable debug logging for the security component: "
            "xpack.security.authc: debug. Check the Elasticsearch logs for messages containing "
            "'authentication failed' or 'realm configuration error'. \n\n"
            "For SAML-specific credential errors: verify the SP metadata is correctly registered "
            "in the IdP. The assertion consumer service URL and entity ID must match exactly. "
            "Clock skew between the ES node and the IdP will cause SAML assertions to be rejected "
            "even when credentials are correct — check that NTP is synchronized."
        )
    },
    {
        "id": "doc-005",
        "title": "LDAP user authentication",
        "url": "https://www.elastic.co/docs/deploy-manage/users-roles/cluster-or-deployment-auth/ldap",
        "body": (
            "LDAP realm configuration requires xpack.security.authc.realms.ldap in elasticsearch.yml. "
            "Common authentication failure causes: incorrect bind DN or bind password, group search "
            "base DN does not match your LDAP schema, user search attribute mismatch. \n\n"
            "To test LDAP connectivity without a full authentication flow, use the Authenticate API: "
            "GET /_security/user/_authenticate with basic auth credentials. If this fails with a "
            "realm configuration error, check your LDAP server connection and the bind credentials."
        )
    },
    {
        "id": "doc-009",
        "title": "Set up security in self-managed Elasticsearch deployments",
        "url": "https://www.elastic.co/docs/deploy-manage/deploy/self-managed/installing-elasticsearch",
        "body": (
            "When security is enabled, all Elasticsearch nodes require TLS for transport and HTTP. "
            "If users cannot authenticate, verify: (1) TLS certificates are valid and not expired, "
            "(2) the xpack.security.enabled setting is true, (3) at least one realm is configured. "
            "The built-in native realm is enabled by default and handles username/password auth. "
            "Check that the user account exists: GET /_security/user/<username>."
        )
    }
]

print(f'Question: {DEMO_QUESTION}')
print(f'Context: {len(GOOD_CONTEXT)} documents (pre-baked good context)')
print('\n--- Answer (pre-generated, same model) ---\n')

GOOD_ANSWER = "Based on the documentation, here are the key things to check when a user can't log in:\n\n**SAML Authentication:**\n1. Check the realm configuration in `elasticsearch.yml` — verify `xpack.security.authc.realms` is correctly set up\n2. Ensure the IdP metadata URL is reachable and returning current metadata\n3. Confirm the principal attribute mapping matches a user in the Elasticsearch user store\n4. Enable debug logging with `xpack.security.authc: debug` and look for 'authentication failed' in the logs\n5. Verify SP metadata is registered in the IdP with matching assertion consumer service URL and entity ID. Check NTP — clock skew rejects valid credentials even when configuration is correct.\n\n**LDAP Authentication:**\n1. Check the bind DN, bind password, and group search base DN in your LDAP realm config\n2. Test with the Authenticate API: `GET /_security/user/_authenticate`\n\n**General:**\n1. Confirm TLS certificates are valid and not expired\n2. Verify `xpack.security.enabled: true` is set\n3. Check the user account exists: `GET /_security/user/<username>`"
print(GOOD_ANSWER)


## Cell 6 — Bad Context Demo (PRE-BAKED)

**Same question. Same model. Same prompt. Different context.**

We replaced the retrieved docs with irrelevant documents about ILM lifecycle policies and snapshot configuration.

### The model didn't get dumber. The retrieval got worse.

This is the core thesis of the workshop. The LLM is a synthesis engine — it works with what you give it. Bad retrieval → bad context → bad answer. The model is not the failure point.

In [ ]:
# Cell 6 — BAD CONTEXT DEMO (pre-baked)
# Same question, same model, same prompt — deliberately wrong context
# DO NOT MODIFY — context is hardcoded to produce a bad answer reproducibly

# Same question as Cell 5
# DEMO_QUESTION = "A user can't log in. What should I check?"

# Deliberately wrong context: ILM and snapshots, completely unrelated to authentication
BAD_CONTEXT = [
    {
        "id": "doc-020",
        "title": "Index Lifecycle Management overview",
        "url": "https://www.elastic.co/docs/manage-data/lifecycle/index-lifecycle-management",
        "body": (
            "Index Lifecycle Management (ILM) automates index management from creation to deletion. "
            "Define a policy with phases: hot (active indexing), warm (read-only, optimized), "
            "cold (infrequent access), frozen (searchable snapshot), delete. "
            "Apply the policy via the index template or directly on the index settings. "
            "Use rollover action to create new indices when the current one reaches a size or age threshold."
        )
    },
    {
        "id": "doc-030",
        "title": "Snapshot and restore overview",
        "url": "https://www.elastic.co/docs/deploy-manage/tools/snapshot-and-restore",
        "body": (
            "Snapshots back up your cluster data to a repository (S3, GCS, Azure, or shared filesystem). "
            "Create a repository: PUT /_snapshot/my_repository. "
            "Take a snapshot: PUT /_snapshot/my_repository/snapshot_1. "
            "Restore: POST /_snapshot/my_repository/snapshot_1/_restore. "
            "Snapshot lifecycle management (SLM) automates scheduled snapshots."
        )
    },
    {
        "id": "doc-040",
        "title": "Ingest pipeline overview",
        "url": "https://www.elastic.co/docs/manage-data/ingest/transform-enrich/ingest-pipelines",
        "body": (
            "Ingest pipelines transform documents before indexing. "
            "A pipeline consists of processors: set, rename, grok, date, remove, script, etc. "
            "Create a pipeline: PUT /_ingest/pipeline/my_pipeline. "
            "Apply to an index via the index's default_pipeline setting or per-request using ?pipeline=. "
            "Use _simulate to test pipeline logic without indexing real documents."
        )
    }
]

print(f'Question: {DEMO_QUESTION}')
print(f'Context: {len(BAD_CONTEXT)} documents (WRONG context — ILM, snapshots, pipelines)')
print('\n--- Answer (pre-generated, same model) ---\n')

BAD_ANSWER = "I don't have information in the provided documentation that directly addresses user login issues. The available documentation covers:\n\n- Index Lifecycle Management (ILM) for automating index management through phases (hot → warm → cold → delete)\n- Snapshot and restore for backing up cluster data to repositories such as S3, GCS, or Azure\n- Ingest pipelines for transforming documents before indexing using processors like set, grok, and date\n\nNone of these topics are relevant to user authentication or login problems. To troubleshoot login issues, you would need documentation about Elasticsearch security configuration, authentication realms, or user management."
print(BAD_ANSWER)

print('\n' + '='*60)
print('THE LESSON: The model did not get dumber. The retrieval got worse.')
print('Retrieval quality determines answer quality.')


In [ ]:
# Cell 7 — Full Pipeline End-to-End  [INSTRUCTOR DEMO — runs on screen]
# Attendees follow along. Take-home: run with SA_LLM_PROXY_BEARER_TOKEN or ANTHROPIC_API_KEY.
#
# TODO (pre-event): confirm LLM_PROXY_URL with ce-infra. SA_LLM_PROXY_BEARER_TOKEN
# is pre-loaded in the managed-vm-elastic-serverless preset secrets.

import anthropic as _anthropic

_proxy_token = os.environ.get('SA_LLM_PROXY_BEARER_TOKEN', '')
_api_key = os.environ.get('ANTHROPIC_API_KEY', '')

if _proxy_token:
    anthropic_client = _anthropic.Anthropic(
        base_url='https://' + os.environ.get('LLM_PROXY_URL', 'litellm-proxy-service-1059491012611.us-central1.run.app'),
        api_key=_proxy_token,
    )
    print('LLM: SA LiteLLM proxy (claude-sonnet-4)')
elif _api_key:
    anthropic_client = _anthropic.Anthropic(api_key=_api_key)
    print('LLM: direct Anthropic API (take-home mode)')
else:
    raise RuntimeError(
        'No LLM credentials found.\n'
        'Workshop: SA_LLM_PROXY_BEARER_TOKEN should be pre-configured in the sandbox.\n'
        'Take-home: export ANTHROPIC_API_KEY=sk-ant-...'
    )


def ask(question: str) -> str:
    """Full RAG pipeline: hybrid retrieve → LLM synthesize."""
    print(f'Retrieving context for: "{question}"')
    docs = hybrid_search(question, size=5)
    print(f'Retrieved {len(docs)} docs:')
    for d in docs:
        print(f'  [{d["id"]}] {d["title"]}')
    print('\nSynthesizing...')
    _model = 'claude-sonnet-4' if _proxy_token else 'claude-haiku-4-5-20251001'
    return synthesize(docs, question, model=_model)


answer = ask("user can't log in")
print('\n--- Answer ---')
print(answer)


In [ ]:
# Cell 7b — Try your own questions  [TAKE-HOME]
# Requires Cell 7 to have run first (anthropic_client must be initialized).

YOUR_QUESTION = "how do I set up TLS for my cluster?"  # Change this

answer = ask(YOUR_QUESTION)
print('\n--- Answer ---')
print(answer)


## Cell 8 — Minimal Multi-Hop Agent Loop (Take-Home)

**This cell is take-home only** — skip it during the workshop.

Requires Cell 7 to have run (needs `anthropic_client` and `ask()`).

A minimal agent that decides when to retrieve again. Each hop uses the prior result.
The point: ES does the heavy lifting on every retrieval hop.


In [ ]:
# Cell 8 — Multi-Hop Agent Loop (TAKE-HOME — skip during workshop)
# The agent retrieves twice — second hop is informed by the first result.

def multi_hop_agent(initial_question: str, max_hops: int = 2) -> str:
    """
    Minimal agent loop: retrieve → synthesize → decide to retrieve again.
    
    Hop 1: retrieve docs for the initial question
    Hop 2: if the answer suggests a follow-up, form a refined query and retrieve again
    """
    print(f'Starting multi-hop agent for: "{initial_question}"')
    all_context = []
    current_question = initial_question
    
    for hop in range(max_hops):
        print(f'\n--- Hop {hop + 1} ---')
        print(f'Query: "{current_question}"')
        
        # Retrieve for current question
        docs = hybrid_search(current_question, size=3)
        print(f'Retrieved: {[d["title"] for d in docs]}')
        all_context.extend(docs)
        
        if hop == max_hops - 1:
            break  # Last hop — synthesize final answer
        
        # Ask the LLM if it needs a follow-up query
        context_text = '\n'.join([f'[{d["title"]}]: {d["body"][:300]}' for d in docs])
        followup_response = anthropic_client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=100,
            system=(
                'You are a search agent. Given a question and initial search results, '
                'decide if a follow-up search query would help answer the question better. '
                'If yes, output ONLY the follow-up query text. '
                'If no follow-up is needed, output: DONE'
            ),
            messages=[{
                'role': 'user',
                'content': f'Question: {initial_question}\n\nInitial results:\n{context_text}\n\nFollow-up query or DONE?'
            }]
        )
        followup = followup_response.content[0].text.strip()
        
        if followup == 'DONE' or not followup:
            print('Agent decided: no follow-up needed')
            break
        else:
            print(f'Agent follow-up query: "{followup}"')
            current_question = followup
    
    # Deduplicate context (by id)
    seen = set()
    unique_context = []
    for doc in all_context:
        if doc['id'] not in seen:
            seen.add(doc['id'])
            unique_context.append(doc)
    
    print(f'\n--- Final synthesis ({len(unique_context)} unique docs) ---')
    return synthesize(unique_context, initial_question)


# Run the multi-hop agent
# This query is designed to benefit from a follow-up: the first hop finds breaking changes,
# the agent then follows up on the specific migration steps
result = multi_hop_agent("authentication issues after upgrading to 9.0")
print('\n=== Final Answer ===')
print(result)